In [ ]:
!pip install transformers datasets accelerate pandas numpy scikit-learn torch

# Import libraries needed

In [ ]:
import pandas as pd
import numpy as np
import os
import re
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizer
from torch.nn import MSELoss, CrossEntropyLoss
import torch
import warnings
warnings.filterwarnings('ignore')

# Connect to my 2 TB drive storage 😎

In [ ]:
# Cloud Storage
from google.cloud import storage
storage_client = storage.Client(project='1Szgj1ZB6PSje2M0hkzdRUhWIFDip0SRQ')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --------------------------------------------------------------
#  Combined script: download + extract + flag reverted PRs
# --------------------------------------------------------------
import io, json, gzip, os, requests
import pandas as pd

# ------------------- 1. Configuration -------------------------
DATE_TO_DOWNLOAD = "2023-01-01"
OUTPUT_CSV = "pull_requests.csv"

base_url = f"https://data.gharchive.org/{DATE_TO_DOWNLOAD}-{{hour}}.json.gz"
urls = [base_url.format(hour=h) for h in range(24)]

# ------------------- 2. Helper containers --------------------
pull_requests = []   # list of dicts – one per PR
reverts = set()      # set of repo names that contain a revert commit

# ------------------- 3. Core processing function ------------
def process_hourly_file(url: str):
    """Download, decompress and parse one hourly .json.gz file."""
    local_prs = []
    local_reverts = set()

    try:
        # 1. Download
        resp = requests.get(url, stream=True, timeout=60)
        resp.raise_for_status()

        # 2. Decompress in memory
        compressed = io.BytesIO(resp.content)
        with gzip.open(compressed, 'rt', encoding='utf-8') as gf:
            for line in gf:
                line = line.strip()
                if not line:
                    continue
                try:
                    event = json.loads(line)
                    ev_type = event.get("type")

                    # ---- PullRequestEvent ----
                    if ev_type == "PullRequestEvent":
                        pr = event["payload"].get("pull_request", {})
                        local_prs.append({
                            "repo": event["repo"]["name"],
                            "author": pr["user"]["login"],
                            "title": pr.get("title"),
                            "lines_added": pr.get("additions"),
                            "lines_deleted": pr.get("deletions"),
                            "files_changed": pr.get("changed_files"),
                            "comments": pr.get("comments"),
                            "commits": pr.get("commits"),
                            "merged": pr.get("merged"),
                            "created_at": pr.get("created_at"),
                            "reverted": False   # placeholder – filled later
                        })

                    # ---- PushEvent (look for "revert" in commit msg) ----
                    elif ev_type == "PushEvent":
                        for commit in event["payload"].get("commits", []):
                            msg = commit.get("message", "").lower()
                            if "revert" in msg:
                                local_reverts.add(event["repo"]["name"])

                except json.JSONDecodeError:
                    continue   # skip malformed line
                except Exception:
                    continue   # any other error – keep going

    except requests.RequestException as e:
        print(f"Network error for {url}: {e}")
    except Exception as e:
        print(f"Unexpected error for {url}: {e}")

    return local_prs, local_reverts


# ------------------- 4. Main loop ---------------------------
print("Starting download & processing of 2023-01-01 (24 files)…")

for i, url in enumerate(urls, 1):
    print(f"[{i:02d}/24] {url}")
    prs, revs = process_hourly_file(url)
    pull_requests.extend(prs)
    reverts.update(revs)
    print(f"   → {len(prs):,} PRs  |  {len(revs):,} new revert-repos")

# ------------------- 5. Build final DataFrame ---------------
print("\nBuilding DataFrame …")
df = pd.DataFrame(pull_requests)

# Mark PRs whose repo appears in the revert set
df["reverted"] = df["repo"].isin(reverts)

# ------------------- 6. Save & preview -----------------------
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nDone! Saved {len(df):,} pull-request rows to '{OUTPUT_CSV}'")
print("\nFirst 5 rows:")
print(df.head())

Starting download & processing of 2023-01-01 (24 files)…
[01/24] https://data.gharchive.org/2023-01-01-0.json.gz
   → 10,440 PRs  |  135 new revert-repos
[02/24] https://data.gharchive.org/2023-01-01-1.json.gz
   → 9,672 PRs  |  121 new revert-repos
[03/24] https://data.gharchive.org/2023-01-01-2.json.gz
   → 8,521 PRs  |  90 new revert-repos
[04/24] https://data.gharchive.org/2023-01-01-3.json.gz
   → 15,047 PRs  |  79 new revert-repos
[05/24] https://data.gharchive.org/2023-01-01-4.json.gz
   → 12,509 PRs  |  79 new revert-repos
[06/24] https://data.gharchive.org/2023-01-01-5.json.gz
   → 10,716 PRs  |  93 new revert-repos
[07/24] https://data.gharchive.org/2023-01-01-6.json.gz
   → 12,500 PRs  |  92 new revert-repos
[08/24] https://data.gharchive.org/2023-01-01-7.json.gz
   → 22,034 PRs  |  108 new revert-repos
[09/24] https://data.gharchive.org/2023-01-01-8.json.gz
   → 21,756 PRs  |  95 new revert-repos
[10/24] https://data.gharchive.org/2023-01-01-9.json.gz
   → 28,804 PRs  |  10

In [ ]:
# Save to a custom folder
custom_path = "/content/drive/MyDrive/Codience/ML Models & Datasets/GP ML part/Phase 2/Datasets/pull_requests.csv"  # Colab + Google Drive
df.to_csv(custom_path, index=False)
print(f"Saved to {custom_path}")

Saved to /content/drive/MyDrive/Codience/ML Models & Datasets/GP ML part/Phase 2/Datasets/pull_requests.csv


In [ ]:
df.columns

Index(['repo', 'author', 'title', 'lines_added', 'lines_deleted',
       'files_changed', 'comments', 'commits', 'merged', 'created_at',
       'reverted'],
      dtype='object')

1Szgj1ZB6PSje2M0hkzdRUhWIFDip0SRQ my drive id

# Label the data

In [ ]:
# ------------------- 2. Features ----------------
df['total_lines'] = df['lines_added'].fillna(0) + df['lines_deleted'].fillna(0)

def norm(col, q_low=0.1, q_high=0.9):
    """Scale column to 0–1 using quantile clipping, safely avoid div-by-zero"""
    low = col.quantile(q_low)
    high = col.quantile(q_high)

    # Clip values
    col_clipped = col.clip(lower=low, upper=high)

    # Compute range
    denominator = high - low
    if denominator == 0:
        return pd.Series([0.5] * len(col), index=col.index)  # neutral score
    else:
        return (col_clipped - low) / denominator

size_score    = norm(df['total_lines'])
spread_score  = norm(df['files_changed'].fillna(0))
commit_score  = norm(df['commits'].fillna(0))
comment_score = 1 - norm(df['comments'].fillna(0))
revert_score  = df['reverted'].astype(int)
merged_score  = (~df['merged'].fillna(False)).astype(int)

weights = {'size':0.30, 'spread':0.20, 'commit':0.15,
           'comment':0.15, 'revert':0.10, 'merged':0.10}

risk_raw = (size_score*weights['size'] +
            spread_score*weights['spread'] +
            commit_score*weights['commit'] +
            comment_score*weights['comment'] +
            revert_score*weights['revert'] +
            merged_score*weights['merged'])

df['risk_score'] = (risk_raw * 100).round().astype(int)

# ------------------- 3. Labels -----------------
def label(s):
    if s <= 25: return "Low"
    if s <= 50: return "Medium"
    if s <= 75: return "High"
    return "Critical"

df['risk_label'] = df['risk_score'].apply(label)

# ------------------- 4. Save -------------------
out = "pull_requests_with_risk.csv"
df.to_csv(out, index=False)
print(f"Saved {len(df):,} rows → {out}")

print("\nRisk distribution:")
print(df['risk_label'].value_counts().sort_index())

print("\nTop 10 riskiest PRs:")
# display(df.nlargest(10, 'risk_score')[['repo','author','title','risk_score','risk_label']])


Saved 520,038 rows → pull_requests_with_risk.csv

Risk distribution:
risk_label
Critical     10332
High        247971
Low          37842
Medium      223893
Name: count, dtype: int64

Top 10 riskiest PRs:


,repo,author,title,risk_score,risk_label
16555,craftycodie/Sunrise-Content,craftycodie,Add more versions.,92,Critical
66273,flexcode-space/log-dispatcher-frontend,Rian3005,Feat/pengaturan user,92,Critical
100084,ProbablyCarl/sunset-wasteland,out-of-phaze,New Year's Faction Update,92,Critical
101243,project-momo/momo-fe,Icyeong,"Revert ""[FE] 모임 등록 테스트 완료""",92,Critical
113353,project-momo/momo-fe,yujinyny,Feat/create content,92,Critical
118804,Arch666Angel/mods,LovelySanta,Version 1.1.2,92,Critical
120623,frosty-dev/ss14,Valtosin,Upstream Merge,92,Critical
127200,Morsmalleo/AhMyth,Morsmalleo,1.0-beta.4,92,Critical
155498,kimxogus/react-native-version-check,kimxogus,update actions,92,Critical
165232,wix/pro-gallery,shmuelhizmi,move-monorepo-to-yarn-3,92,Critical


In [ ]:
df.head()

,repo,author,title,lines_added,lines_deleted,files_changed,comments,commits,merged,created_at,reverted,total_lines,risk_score,risk_label
0,sonusathyadas/musician-app,dependabot[bot],Bump json5 and react-scripts in /client,14363,16201,3,0,1,False,2023-01-01T00:00:00Z,False,30564,82,Critical
1,mhaji007/Clue,dependabot[bot],Bump axios from 0.19.2 to 0.21.1 in /server/cl...,5,28,2,1,1,False,2021-01-06T02:45:39Z,False,33,28,Medium
2,fulpstation/fulpstation,FernandoJ8,January TGU - First Draft,52918,29315,1007,0,7,False,2023-01-01T00:00:00Z,False,82233,82,Critical
3,mhaji007/Clue,dependabot[bot],Bump terser from 4.6.13 to 4.8.1 in /server/cl...,3,3,1,1,1,False,2022-12-20T01:29:38Z,False,6,18,Low
4,mhaji007/Clue,dependabot[bot],Bump ansi-regex from 4.1.0 to 4.1.1 in /server...,13,10,1,0,1,False,2023-01-01T00:00:03Z,False,23,33,Medium


In [ ]:
# === 3. Save to Google Drive ===
custom_path = "/content/drive/MyDrive/Codience/ML Models & Datasets/GP ML part/Phase 2/Datasets/pull_requests_labeled.csv"  # Colab + Google Drive
df.to_csv(custom_path, index=False)

In [ ]:
df.columns

Index(['repo', 'author', 'title', 'lines_added', 'lines_deleted',
       'files_changed', 'comments', 'commits', 'merged', 'created_at',
       'reverted', 'total_lines', 'risk_score', 'risk_label'],
      dtype='object')

# Testing upload to drive

# Train the model (not working yet)

In [ ]:
# ==============================================================================
# PR RISK REGRESSION: Tiny BERT (title) + Numeric Features ONLY
# INPUT: title, lines_added, lines_deleted, files_changed, comments,
#        commits, merged, reverted, total_lines
# TARGET: risk_score
# REMOVED: repo, author, created_at (time)
# ==============================================================================

# -------------------------------------------------
# 1. Mount Drive & Install
# -------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers torch scikit-learn pandas numpy

# -------------------------------------------------
# 2. Imports
# -------------------------------------------------
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os, pickle

# -------------------------------------------------
# 3. CONFIG
# -------------------------------------------------
INPUT_CSV = '/content/drive/MyDrive/Codience/ML Models & Datasets/Risk Analysis/Dataset/Datasets/pull_requests_labeled.csv'
MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"
MAX_SEQ_LEN = 64
BATCH_SIZE = 64
LR = 5e-5
EPOCHS = 5

DRIVE_OUT = '/content/drive/MyDrive/Codience/ML Models & Datasets/Risk Analysis/Risk Model/BERT_Tiny_Minimal'
os.makedirs(DRIVE_OUT, exist_ok=True)

BEST_CKPT = 'bert_minimal_best.pth'
FINAL_CKPT = 'bert_minimal_final.pth'

GLOBAL_BEST = float('inf')

# -------------------------------------------------
# 4. Load & Pre-process (NO time, repo, author)
# -------------------------------------------------
print("Loading data...")
df = pd.read_csv(INPUT_CSV)

# Keep only these
num_cols = ['lines_added','lines_deleted','files_changed','comments',
            'commits','total_lines','reverted','merged']
text_col = 'title'
target = 'risk_score'

# Fill missing
df[num_cols] = df[num_cols].fillna(0)
df[text_col] = df[text_col].fillna('')

# Drop NaN target
df = df.dropna(subset=[target])

# Scale numeric
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# Save scaler
with open(os.path.join(DRIVE_OUT, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

# Split
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,}")

# -------------------------------------------------
# 5. Dataset
# -------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PRDataset(Dataset):
    def __init__(self, df):
        self.texts = df[text_col].tolist()
        self.nums  = df[num_cols].values.astype(np.float32)
        self.y     = df[target].values.astype(np.float32)

    def __len__(self): return len(self.y)

    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]),
            truncation=True, padding='max_length',
            max_length=MAX_SEQ_LEN, return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'num_feats': torch.tensor(self.nums[idx]),
            'label': torch.tensor(self.y[idx])
        }

train_ds = PRDataset(train_df)
val_ds = PRDataset(val_df)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE)

# -------------------------------------------------
# 6. Model: Tiny BERT + Numeric ONLY
# -------------------------------------------------
class RiskBERTMinimal(nn.Module):
    def __init__(self, bert_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_name)
        bert_dim = self.bert.config.hidden_size  # 128
        self.num_fc = nn.Linear(len(num_cols), 64)
        self.head = nn.Sequential(
            nn.Linear(bert_dim + 64, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, num_feats):
        bert_out = self.bert(input_ids, attention_mask).pooler_output
        num_out = torch.relu(self.num_fc(num_feats))
        x = torch.cat([bert_out, num_out], dim=1)
        return self.head(x).squeeze(-1)

model = RiskBERTMinimal(MODEL_NAME)
criterion = nn.MSELoss()

# -------------------------------------------------
# 7. Training Utils
# -------------------------------------------------
def save_ckpt(model, opt, epoch, loss, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'loss': loss
    }, path)
    print(f"Saved: {path}")

def load_ckpt(model, opt, path):
    global GLOBAL_BEST
    if not os.path.exists(path):
        print("No checkpoint → start fresh")
        return 0
    ck = torch.load(path, map_location='cpu')
    model.load_state_dict(ck['model_state_dict'])
    opt.load_state_dict(ck['optimizer_state_dict'])
    GLOBAL_BEST = ck['loss']
    print(f"Resumed from epoch {ck['epoch']+1}, best={GLOBAL_BEST:.4f}")
    return ck['epoch'] + 1

# -------------------------------------------------
# 8. Train + Evaluate
# -------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

start_epoch = load_ckpt(model, optimizer, os.path.join(DRIVE_OUT, BEST_CKPT))

for epoch in range(start_epoch, EPOCHS):
    model.train()
    tr_loss = 0
    for i, batch in enumerate(train_dl):
        optimizer.zero_grad()
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        num = batch['num_feats'].to(device)
        y = batch['label'].to(device)

        pred = model(ids, mask, num)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()

        if (i+1) % 100 == 0:
            print(f"Epoch {epoch+1} | Step {i+1} | Loss: {loss.item():.4f}")

    # Validation
    model.eval()
    val_loss = 0
    all_preds = []
    all_true = []

    with torch.no_grad():
        for batch in val_dl:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            num = batch['num_feats'].to(device)
            y = batch['label'].cpu().numpy()

            pred = model(ids, mask, num).cpu().numpy()
            all_preds.extend(pred)
            all_true.extend(y)
            val_loss += criterion(torch.tensor(pred).to(device), torch.tensor(y).to(device)).item()

    avg_tr = tr_loss / len(train_dl)
    avg_val = val_loss / len(val_dl)

    # Accuracy metrics
    all_preds = np.clip(all_preds, 0, 100)
    mae = mean_absolute_error(all_true, all_preds)
    rmse = np.sqrt(mean_squared_error(all_true, all_preds))
    r2 = r2_score(all_true, all_preds)
    within_5 = np.mean(np.abs(np.array(all_true) - all_preds) <= 5) * 100
    within_10 = np.mean(np.abs(np.array(all_true) - all_preds) <= 10) * 100

    print(f"\n=== Epoch {epoch+1}/{EPOCHS} ===")
    print(f"Train Loss: {avg_tr:.4f} | Val Loss: {avg_val:.4f}")
    print(f"MAE: {mae:.3f} | RMSE: {rmse:.3f} | R²: {r2:.3f}")
    print(f"±5 Acc: {within_5:.1f}% | ±10 Acc: {within_10:.1f}%")

    if avg_val < GLOBAL_BEST:
        GLOBAL_BEST = avg_val
        save_ckpt(model, optimizer, epoch, avg_val, os.path.join(DRIVE_OUT, BEST_CKPT))

# Final save
torch.save(model.state_dict(), os.path.join(DRIVE_OUT, FINAL_CKPT))
print(f"\nTraining done! Final model → {FINAL_CKPT}")

# Save validation predictions
val_df = val_df.copy()
val_df['pred_risk_score'] = np.clip(all_preds, 0, 100).round().astype(int)
val_df[['title', 'risk_score', 'pred_risk_score']].to_csv(
    os.path.join(DRIVE_OUT, 'validation_predictions.csv'), index=False
)
print("Validation predictions saved!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training: 442,032 | Validation: 78,006


config.json:   0%|          | 0.00/382 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Using device: cpu
No checkpoint → start fresh
Epoch 1 | Step 100 | Loss: 2576.8542
Epoch 1 | Step 200 | Loss: 1910.8759
Epoch 1 | Step 300 | Loss: 1630.3561


KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# FINAL PR RISK MODEL: Tiny BERT + 7 Numeric Features
# INPUT: title, lines_added, lines_deleted, files_changed, comments,
#        commits, merged, reverted
# TARGET: risk_score (0–100)
# REMOVED: total_lines, repo, author, created_at
# ==============================================================================

# -------------------------------------------------
# 1. Mount Drive & Install
# -------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers torch scikit-learn pandas numpy

# -------------------------------------------------
# 2. Imports
# -------------------------------------------------
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os, pickle

# -------------------------------------------------
# 3. CONFIG
# -------------------------------------------------
INPUT_CSV = '/content/drive/MyDrive/Codience/ML Models & Datasets/Risk Analysis/Dataset/Datasets/pull_requests_labeled.csv'
MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"
MAX_SEQ_LEN = 64
BATCH_SIZE = 64
LR = 5e-5
EPOCHS = 5

DRIVE_OUT = '/content/drive/MyDrive/Codience/ML Models & Datasets/Risk Analysis/Risk Model/BERT_Final_Clean'
os.makedirs(DRIVE_OUT, exist_ok=True)

BEST_CKPT = 'bert_final_best.pth'
FINAL_CKPT = 'bert_final_model.pth'
SCALER_PATH = os.path.join(DRIVE_OUT, 'scaler.pkl')

GLOBAL_BEST = float('inf')

# -------------------------------------------------
# 4. Load & Pre-process (CLEAN & MINIMAL)
# -------------------------------------------------
print("Loading data...")
df = pd.read_csv(INPUT_CSV)

# Essential features only
num_cols = ['lines_added', 'lines_deleted', 'files_changed', 'comments',
            'commits', 'reverted', 'merged']  # 7 features
text_col = 'title'
target = 'risk_score'

# Clean
df[num_cols] = df[num_cols].fillna(0)
df[text_col] = df[text_col].fillna('')
df = df.dropna(subset=[target])

# Scale numeric
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# Save scaler
with open(SCALER_PATH, 'wb') as f:
    pickle.dump(scaler, f)

# Split
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,}")

# -------------------------------------------------
# 5. Dataset
# -------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PRDataset(Dataset):
    def __init__(self, df):
        self.texts = df[text_col].tolist()
        self.nums  = df[num_cols].values.astype(np.float32)
        self.y     = df[target].values.astype(np.float32)

    def __len__(self): return len(self.y)

    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=MAX_SEQ_LEN,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'num_feats': torch.tensor(self.nums[idx]),
            'label': torch.tensor(self.y[idx])
        }

train_ds = PRDataset(train_df)
val_ds = PRDataset(val_df)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE)

# -------------------------------------------------
# 6. Model: Tiny BERT + 7 Numerics
# -------------------------------------------------
class RiskBERTFinal(nn.Module):
    def __init__(self, bert_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_name)
        bert_dim = self.bert.config.hidden_size  # 128
        self.num_fc = nn.Linear(7, 64)  # 7 numeric features
        self.head = nn.Sequential(
            nn.Linear(bert_dim + 64, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, num_feats):
        bert_out = self.bert(input_ids, attention_mask).pooler_output
        num_out = torch.relu(self.num_fc(num_feats))
        x = torch.cat([bert_out, num_out], dim=1)
        return self.head(x).squeeze(-1)

model = RiskBERTFinal(MODEL_NAME)
criterion = nn.MSELoss()

# -------------------------------------------------
# 7. Training Utils
# -------------------------------------------------
def save_ckpt(model, opt, epoch, loss, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'loss': loss
    }, path)
    print(f"Best model saved: {path}")

def load_ckpt(model, opt, path):
    global GLOBAL_BEST
    if not os.path.exists(path):
        print("No checkpoint → training from scratch")
        return 0
    ck = torch.load(path, map_location='cpu')
    model.load_state_dict(ck['model_state_dict'])
    opt.load_state_dict(ck['optimizer_state_dict'])
    GLOBAL_BEST = ck['loss']
    print(f"Resumed from epoch {ck['epoch']+1}, best val loss: {GLOBAL_BEST:.4f}")
    return ck['epoch'] + 1

# -------------------------------------------------
# 8. Train + Evaluate
# -------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

start_epoch = load_ckpt(model, optimizer, os.path.join(DRIVE_OUT, BEST_CKPT))

for epoch in range(start_epoch, EPOCHS):
    model.train()
    tr_loss = 0
    for i, batch in enumerate(train_dl):
        optimizer.zero_grad()
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        num = batch['num_feats'].to(device)
        y = batch['label'].to(device)

        pred = model(ids, mask, num)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()

        if (i + 1) % 100 == 0:
            print(f"Epoch {epoch+1} | Step {i+1}/{len(train_dl)} | Loss: {loss.item():.4f}")

    # Validation
    model.eval()
    val_loss = 0
    all_preds = []
    all_true = []

    with torch.no_grad():
        for batch in val_dl:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            num = batch['num_feats'].to(device)
            y = batch['label'].cpu().numpy()

            pred = model(ids, mask, num).cpu().numpy()
            all_preds.extend(pred)
            all_true.extend(y)
            val_loss += criterion(torch.tensor(pred).to(device), torch.tensor(y).to(device)).item()

    avg_tr = tr_loss / len(train_dl)
    avg_val = val_loss / len(val_dl)

    # Accuracy
    all_preds = np.clip(all_preds, 0, 100)
    mae = mean_absolute_error(all_true, all_preds)
    rmse = np.sqrt(mean_squared_error(all_true, all_preds))
    r2 = r2_score(all_true, all_preds)
    within_5 = np.mean(np.abs(np.array(all_true) - all_preds) <= 5) * 100
    within_10 = np.mean(np.abs(np.array(all_true) - all_preds) <= 10) * 100

    print(f"\n{'='*60}")
    print(f" EPOCH {epoch+1}/{EPOCHS} COMPLETE")
    print(f"{'='*60}")
    print(f"Train Loss: {avg_tr:.4f} | Val Loss: {avg_val:.4f}")
    print(f"MAE: {mae:.3f} | RMSE: {rmse:.3f} | R²: {r2:.3f}")
    print(f"±5 Accuracy:  {within_5:.1f}%")
    print(f"±10 Accuracy: {within_10:.1f}%")
    print(f"{'='*60}\n")

    if avg_val < GLOBAL_BEST:
        GLOBAL_BEST = avg_val
        save_ckpt(model, optimizer, epoch, avg_val, os.path.join(DRIVE_OUT, BEST_CKPT))

# Final save
torch.save(model.state_dict(), os.path.join(DRIVE_OUT, FINAL_CKPT))
print(f"Final model saved: {FINAL_CKPT}")

# Save predictions
val_df = val_df.copy()
val_df['pred_risk_score'] = np.clip(all_preds, 0, 100).round().astype(int)
val_df[['title', 'risk_score', 'pred_risk_score']].to_csv(
    os.path.join(DRIVE_OUT, 'validation_predictions.csv'), index=False
)
print("Validation predictions saved!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading data...
Train: 442,032 | Val: 78,006
No checkpoint → training from scratch
Epoch 1 | Step 100/6907 | Loss: 2417.9302
Epoch 1 | Step 200/6907 | Loss: 1929.8556
Epoch 1 | Step 300/6907 | Loss: 1778.9519
Epoch 1 | Step 400/6907 | Loss: 1290.0149
Epoch 1 | Step 500/6907 | Loss: 1149.7733
Epoch 1 | Step 600/6907 | Loss: 613.5498
Epoch 1 | Step 700/6907 | Loss: 507.5052
Epoch 1 | Step 800/6907 | Loss: 395.9871
Epoch 1 | Step 900/6907 | Loss: 315.1963
Epoch 1 | Step 1000/6907 | Loss: 348.4642
Epoch 1 | Step 1100/6907 | Loss: 391.0738
Epoch 1 | Step 1200/6907 | Loss: 332.0929
Epoch 1 | Step 1300/6907 | Loss: 170.7220
Epoch 1 | Step 1400/6907 | Loss: 121.5493
Epoch 1 | Step 1500/6907 | Loss: 129.1191
Epoch 1 | Step 1600/6907 | Loss: 53.3989
Epoch 1 | Step 1700/6907 | Loss: 96.6738
Epoch 1 | Step 1800/6907 | Loss: 69.9507
Epoch 1 | Step 1900/6907 | Loss: 47.407

# Train 2

In [ ]:
!pip install -q transformers torch scikit-learn pandas numpy tqdm


In [ ]:
# ==============================================================
# FULL TRAINING SCRIPT — RUN THIS TO TRAIN & SAVE MODEL
# ==============================================================
# 0. INSTALL (run once)
# !pip install torch transformers scikit-learn pandas numpy tqdm

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import os
import pickle

# -------------------------------------------------
# 1. LOAD OR CREATE DATA
# -------------------------------------------------
# OPTION A: Use your real CSV
# df = pd.read_csv('/content/pull_requests_labeled.csv')

# # OPTION B: Use small sample (uncomment if no data)
# data = [
#     ("Fix typo", 1, 1, 1, 0, 1, True, False, 5),
#     ("Refactor auth", 120, 80, 5, 3, 4, True, False, 52),
#     ("Fix SQL injection", 180, 60, 8, 5, 6, False, False, 82),
#     ("Update README", 2, 1, 1, 0, 1, True, False, 6),
#     ("DB migration", 300, 150, 12, 6, 8, False, False, 88),
# ]
df = pd.read_csv("/content/drive/MyDrive/Codience/ML Models & Datasets/Risk Analysis/Dataset/Datasets/pull_requests_labeled.csv")
print(f"Loaded {len(df)} PRs")

# -------------------------------------------------
# 2. LABEL WITH HEURISTIC (FIXED: no ~ on object dtype)
# -------------------------------------------------
df['total_lines'] = df['lines_added'] + df['lines_deleted']

def norm(col):
    low = col.quantile(0.1)
    high = col.quantile(0.9)
    clipped = col.clip(lower=low, upper=high)
    return (clipped - low) / (high - low + 1e-8)

size_score = norm(df['total_lines'])
spread_score = norm(df['files_changed'])
commit_score = norm(df['commits'])
comment_score = 1 - norm(df['comments'])
revert_score = df['reverted'].fillna(False).astype(int)

# FIXED: Avoid ~ on potentially object dtype → use comparison
merged_score = (df['merged'] != True).astype(int)  # NaN or False → 1 (risky)

risk_raw = (
    size_score * 0.3 +
    spread_score * 0.2 +
    commit_score * 0.15 +
    comment_score * 0.15 +
    revert_score * 0.1 +
    merged_score * 0.1
)
df['risk_score'] = (risk_raw * 100).round().astype(int)
print(f"Labeled {len(df)} PRs")

# -------------------------------------------------
# 3. PREPARE FEATURES
# -------------------------------------------------
num_cols = ['lines_added', 'lines_deleted', 'files_changed', 'comments',
            'commits', 'reverted', 'merged']
df[num_cols] = df[num_cols].fillna(0)
df = df.dropna(subset=['risk_score', 'title'])

# Scale numeric features
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# Keyword features
security_kw = {"security", "vulnerability", "auth", "login", "password",
               "token", "payment", "exploit", "xss", "sqli"}
db_kw = {"database", "schema", "migration", "table"}

def add_features(row):
    t = str(row['title']).lower()
    total = row['lines_added'] + row['lines_deleted']
    files = max(1, row['files_changed'])
    return pd.Series({
        'total_changes': total,
        'change_density': total / files,
        'commit_density': row['commits'] / files,
        'is_security': int(any(k in t for k in security_kw)),
        'is_database': int(any(k in t for k in db_kw))
    })

df[['total_changes', 'change_density', 'commit_density',
    'is_security', 'is_database']] = df.apply(add_features, axis=1)

# Final numeric features
final_num_cols = num_cols + ['total_changes', 'change_density',
                            'commit_density', 'is_security', 'is_database']
X_num = df[final_num_cols].values.astype(np.float32)
y = df['risk_score'].values.astype(np.float32) / 100.0  # Normalize to [0,1]

# -------------------------------------------------
# 4. BERT EMBEDDINGS (efficient batched version)
# -------------------------------------------------
MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.eval()

@torch.no_grad()
def get_bert_embeddings(titles, batch_size=16):
    embeddings = []
    for i in tqdm(range(0, len(titles), batch_size), desc="BERT Embedding"):
        batch = titles[i:i+batch_size]
        enc = tokenizer(
            batch, truncation=True, padding=True,
            max_length=64, return_tensors='pt'
        )
        enc = {k: v.to(bert_model.device) for k, v in enc.items()}
        out = bert_model(**enc).pooler_output
        embeddings.append(out.cpu().numpy())
    return np.concatenate(embeddings, axis=0)

print("Extracting BERT features...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.to(device)

titles = df['title'].tolist()
X_text = get_bert_embeddings(titles)

# -------------------------------------------------
# 5. TRAIN-TEST SPLIT & DATASET
# -------------------------------------------------
X_text_tr, X_text_val, X_num_tr, X_num_val, y_tr, y_val = train_test_split(
    X_text, X_num, y, test_size=0.2, random_state=42
)

class PRDataset(Dataset):
    def __init__(self, txt, num, y):
        self.txt = torch.FloatTensor(txt)
        self.num = torch.FloatTensor(num)
        self.y = torch.FloatTensor(y)

    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.txt[i], self.num[i], self.y[i]

train_ds = PRDataset(X_text_tr, X_num_tr, y_tr)
val_ds = PRDataset(X_text_val, X_num_val, y_val)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=32)

# -------------------------------------------------
# 6. MODEL (FIXED: input size = 12 numeric features)
# -------------------------------------------------
class RiskBERT12(nn.Module):
    def __init__(self, bert_dim=128, num_features=12):
        super().__init__()
        self.num_fc = nn.Linear(num_features, 64)
        self.head = nn.Sequential(
            nn.Linear(bert_dim + 64, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, txt, num):
        num_out = torch.relu(self.num_fc(num))
        x = torch.cat([txt, num_out], dim=1)
        return self.head(x).squeeze(-1)

model = RiskBERT12(bert_dim=128, num_features=len(final_num_cols))
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
criterion = nn.MSELoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print("Training model...")
for epoch in range(8):
    model.train()
    tr_loss = 0
    for txt, num, y in train_dl:
        txt, num, y = txt.to(device), num.to(device), y.to(device)
        pred = model(txt, num)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for txt, num, y in val_dl:
            txt, num, y = txt.to(device), num.to(device), y.to(device)
            pred = model(txt, num)
            val_loss += criterion(pred, y).item()

    print(f"Epoch {epoch+1}/8 | Train: {tr_loss/len(train_dl):.4f} | Val: {val_loss/len(val_dl):.4f}")

# -------------------------------------------------
# 7. SAVE MODEL & SCALER
# -------------------------------------------------
SAVE_DIR = "/content/drive/MyDrive/RiskModelFinal/trained_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save model + scaler
torch.save({
    'model_state_dict': model.state_dict(),
    'scaler': scaler,
    'num_features': final_num_cols,
    'bert_model_name': MODEL_NAME
}, os.path.join(SAVE_DIR, 'model_and_scaler.pth'))

with open(os.path.join(SAVE_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

print(f"MODEL SAVED TO: {SAVE_DIR}")
print(" → model_and_scaler.pth")
print(" → scaler.pkl")

Loaded 520038 PRs
Labeled 520038 PRs
Extracting BERT features...


BERT Embedding: 100%|██████████| 32503/32503 [13:12<00:00, 41.04it/s]


Training model...
Epoch 1/8 | Train: 0.0032 | Val: 0.0021
Epoch 2/8 | Train: 0.0020 | Val: 0.0019
Epoch 3/8 | Train: 0.0018 | Val: 0.0017
Epoch 4/8 | Train: 0.0017 | Val: 0.0015
Epoch 5/8 | Train: 0.0015 | Val: 0.0014
Epoch 6/8 | Train: 0.0014 | Val: 0.0013
Epoch 7/8 | Train: 0.0012 | Val: 0.0011
Epoch 8/8 | Train: 0.0011 | Val: 0.0009
MODEL SAVED TO: /content/drive/MyDrive/RiskModelFinal/trained_model
 → model_and_scaler.pth
 → scaler.pkl


In [ ]:
ls /content/MyDrive/RiskModelFinal/trained_model

model_and_scaler.pth  scaler.pkl


#FASTAPI

In [ ]:
# -------------------------------------------------
# 1. Install dependencies
# -------------------------------------------------
!pip install -q fastapi uvicorn[standard] python-multipart \
               torch transformers scikit-learn pandas numpy \
               pyngrok nest-asyncio requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.7/517.7 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.8/456.8 kB 24.0 MB/s eta 0:00:00


In [ ]:
!ngrok authtoken 34pSpw55HeyCnVVcNVv5reKacXe_4VCGDVFdoyxrqsjbGTQSx

/bin/bash: line 1: ngrok: command not found


In [ ]:
import shutil
import os

# Source and target
src_file = "/content/MyDrive/RiskModelFinal/trained_model/"
dst_file = "/content/drive/MyDrive/RiskModelFinal/trained_model/"

# Ensure target folder exists
os.makedirs(os.path.dirname(dst_file), exist_ok=True)

# Move file
shutil.move(src_file, dst_file)
print("File moved successfully!")


File moved successfully!
